# Test Roman's LoRA on Kaggle (free likeness test)

Runs unattended with **Save Version → Save & Run All**. It either finishes or stops early with
the reason as the last line and in `status.json`.

- **Input:** the chosen LoRA (attach the training notebook's output: its folder or its
  `*_outputs.zip`, or a dataset with the `.safetensors` file) **and** the private dataset from
  `ca lora prepare`. Set `LORA_FILE` below; leave it empty for the final file (else the highest
  step).
- **Settings:** Accelerator **GPU T4 x2** (only GPU 0 is used), Internet **on**.
- **Same environment as training:** the pinned AI Toolkit commit and packages, bf16 for the
  model, text encoder and VAE (nothing in fp16), no bitsandbytes. The images are made by AI
  Toolkit's own sampler, the code that made the training samples, so the LoRA is applied
  exactly as in training. The learning rate is 0 and no training step runs.
- **Smoke check first:** one image at 512 px (exists, not blank). Then the test set: 6 fixed
  prompts × LoRA strength 0.7 / 0.9 / 1.0, the same seed per prompt, Z-Image Turbo at 8 steps
  without CFG, 1024 × 1024.
- **Output (`/kaggle/working/lora_test/`):** `contact_sheet.jpg` (rows = prompts, columns =
  strengths), the 18 images, `settings.json`, `status.json`, all in `lora_test.zip`.

Compare each image with Roman's photos: same face, hairline, build and age?

In [ ]:
import time

LORA_FILE = ""  # e.g. "r0man_bg_z_image_turbo_000001000.safetensors"; empty = the final file
STRENGTHS = [0.7, 0.9, 1.0]
SIZE = 1024
SMOKE_SIZE = 512
PROMPTS = {  # name -> (prompt, fixed seed); [trigger] is the LoRA's trigger word
    "office portrait": ("[trigger], a man in a navy suit in a bright modern office, "
                        "professional portrait, soft window light, sharp focus", 1001),
    "facebook ad": ("[trigger], a man standing on the right side of the frame in a modern tech "
                    "office with blue accent light, the left half is calm, empty dark background "
                    "for text, advertising photo", 1002),
    "outdoor": ("[trigger], a man walking in a city street on a sunny day, smart casual jacket, "
                "natural light, photo", 1003),
    "side angle": ("[trigger], side profile of a man looking out of a window, three-quarter "
                   "view, soft light, photo", 1004),
    "smiling": ("[trigger], a man smiling warmly at the camera, close-up portrait, natural "
                "light", 1005),
    "linkedin": ("[trigger], professional LinkedIn headshot of a man, light grey background, "
                 "studio light, white shirt and dark blazer", 1006),
}
SMOKE_MINUTES = 20  # model load + caching + 1 image
FULL_MINUTES = 150  # model load + 18 images at 1024 px (~4 min each on a T4)
MIN_IMAGES = 9
RESOLUTIONS = [512]  # the trainer caches the dataset at this size; nothing is trained
LEARNING_RATE = 0.0  # no training: the LoRA cannot change
OPTIMIZER = "adamw"
KEEP_CHECKPOINTS = 4

WORK = "/kaggle/working"
OUTPUT_DIR = f"{WORK}/lora_test"
SCRATCH = "/tmp/ca_lora_test"  # dataset copy, toolkit, base models and runs: not saved

# AI Toolkit at a fixed commit (never "latest"); every config key below was checked against
# its config code at this commit.
TOOLKIT_REPO = "https://github.com/ostris/ai-toolkit.git"
TOOLKIT_COMMIT = "f7a1fb9a77e5418269b1e9eda4ace6d7c1b7bb47"
BASE_MODEL = "Tongyi-MAI/Z-Image-Turbo"
ADAPTER_REPO = "ostris/zimage_turbo_training_adapter"
ADAPTER_FILE = "zimage_turbo_training_adapter_v2.safetensors"

# Packages: AI Toolkit's requirements at TOOLKIT_COMMIT (+ its "-r" include), each pinned to a
# version with a Python 3.13 manylinux wheel (checked on PyPI on 2026-10-10). Exceptions:
# lycoris-lora 1.8.3 is pure Python without a wheel (it installed from source on Kaggle; the
# "nearest" wheel is 3.3.0, a risky major jump); scipy 1.12.0 has no 3.13 wheel -> 1.14.1;
# opencv stays below 5 (5.0 removed APIs). Unpinned requirements: the newest version with a
# 3.13 wheel released on or before the commit date (2026-10-09).
SKIP_PACKAGES = {
    "torch", "torchvision", "torchaudio",  # Kaggle's CUDA build stays
    "bitsandbytes",  # only the 8-bit optimizers import it; it is uninstalled
    "torchcodec",  # audio only; built against one torch version
    "gradio", "fastapi", "uvicorn",  # UI and inference server, not training
}
PINNED: dict[str, str] = {
    "torchao": "torchao==0.10.0",
    "safetensors": "safetensors==0.8.0",
    "transformers": "transformers==5.5.3",
    "lycoris-lora": "lycoris-lora==1.8.3",
    "flatten-json": "flatten_json==0.1.14",
    "pyyaml": "pyyaml==6.0.3",
    "oyaml": "oyaml==1.0",
    "tensorboard": "tensorboard==2.21.0",
    "kornia": "kornia==0.8.3",
    "invisible-watermark": "invisible-watermark==0.2.0",
    "einops": "einops==0.8.2",
    "accelerate": "accelerate==1.15.0",
    "toml": "toml==0.10.2",
    "albumentations": "albumentations==1.4.15",
    "albucore": "albucore==0.0.16",
    "pydantic": "pydantic==2.14.0",
    "omegaconf": "omegaconf==2.4.0",
    "open-clip-torch": "open_clip_torch==3.3.0",
    "timm": "timm==1.0.22",
    "prodigyopt": "prodigyopt==1.1.2",
    "controlnet-aux": "controlnet_aux==0.0.10",
    "python-dotenv": "python-dotenv==1.2.4",
    "hf-transfer": "hf_transfer==0.1.9",
    "lpips": "lpips==0.1.4",
    "pytorch-fid": "pytorch_fid==0.3.0",
    "optimum-quanto": "optimum-quanto==0.2.4",
    "sentencepiece": "sentencepiece==0.2.2",
    "huggingface-hub": "huggingface_hub==1.23.0",
    "peft": "peft==0.18.1",
    "python-slugify": "python-slugify==9.1.3",
    "opencv-python": "opencv-python==4.14.0.94",
    "pytorch-wavelets": "pytorch-wavelets==1.3.0",
    "matplotlib": "matplotlib==3.10.1",
    "setuptools": "setuptools==84.0.0",
    "av": "av==16.0.1",
    "librosa": "librosa==0.11.0",
    "mutagen": "mutagen==1.47.0",
    "scipy": "scipy==1.14.1",
}
EXTRA_PINS = ["opencv-python-headless==4.14.0.94"]  # albumentations' opencv: also below 5

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import NoReturn

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU only (T4 x2 has two); before torch
import torch  # noqa: E402 - after CUDA_VISIBLE_DEVICES

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)
os.environ["HF_HOME"] = f"{SCRATCH}/hf"  # base models stay out of /kaggle/working
# The trainer runs as a subprocess: one GPU, and a segfault prints the Python stack of every
# thread into its log.
TRAIN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONFAULTHANDLER": "1"}
STATUS: dict[str, object] = {"status": "started"}


def save_status(**changes: object) -> None:
    STATUS.update(changes)
    Path(OUTPUT_DIR, "status.json").write_text(json.dumps(STATUS, indent=2))


def fail(reason: str) -> NoReturn:
    """Stop the notebook: the reason goes to status.json and is the last line printed."""
    save_status(status=f"failed: {reason}")
    raise SystemExit(f"FAILED: {reason}")


def sh(command: str) -> None:
    """A setup command; a failure stops the notebook with its reason."""
    print("$", command, flush=True)
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    if done.returncode != 0:
        fail(f"{command!r} failed: {(done.stdout + done.stderr).strip()[-600:]}")


if not torch.cuda.is_available():
    fail("no GPU: Settings -> Accelerator -> GPU T4 x2, then Save & Run All again")
if torch.cuda.device_count() != 1:
    fail(f"{torch.cuda.device_count()} GPUs visible; CUDA_VISIBLE_DEVICES=0 did not apply")
MAJOR, MINOR = torch.cuda.get_device_capability(0)


def bf16_works() -> bool:
    """bf16 matmul, conv and softmax on this GPU give finite results close to fp32."""
    try:
        torch.manual_seed(0)
        a = torch.randn(512, 512, device="cuda")
        image = torch.randn(1, 16, 64, 64, device="cuda")
        kernel = torch.randn(32, 16, 3, 3, device="cuda")
        product = (a.bfloat16() @ a.bfloat16()).float()
        conv = torch.nn.functional.conv2d(image.bfloat16(), kernel.bfloat16()).float()
        soft = torch.softmax(a.bfloat16() * 30, dim=-1).float()
        close = torch.allclose(product, a @ a, rtol=0.05, atol=2.0)
        return bool(torch.isfinite(product).all() and torch.isfinite(conv).all()
                    and torch.isfinite(soft).all() and close)
    except RuntimeError as error:
        print("bf16 test failed:", error)
        return False


# Never fp16 (Z-Image overflows it): bf16 is native from compute 8.0 and emulated on the
# T4/P100 (slower but correct). If bf16 does not work at all, nothing would train correctly.
if MAJOR < 8 and not bf16_works():
    fail("bf16 does not work on this GPU and fp16 overflows Z-Image")
DTYPE = "bf16"
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU 0: {GPU_NAME} (compute {MAJOR}.{MINOR}), "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB, dtype {DTYPE}"
      + ("" if MAJOR >= 8 else " (emulated on this GPU: slower)"))
save_status(dtype=DTYPE, gpu=GPU_NAME)

In [ ]:
# The dataset from `ca lora prepare`, found wherever Kaggle put it (unpacked or still zipped,
# any folder depth, any image-extension case). A missing or untidy caption is fixed: every
# caption starts with the trigger word, the most common first word of the captions.
import shutil
import zipfile
from collections import Counter

IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".webp"}
OWN_OUTPUT = {"lora_output", "smoke_test"}  # an attached earlier output is not a dataset


def images_in(folder: Path) -> list[Path]:
    return sorted(path for path in folder.iterdir()
                  if path.is_file() and path.suffix.lower() in IMAGE_TYPES)


def caption_for(image: Path) -> str | None:
    for path in image.parent.iterdir():
        if path.suffix.lower() == ".txt" and path.stem.lower() == image.stem.lower():
            return path.read_text(encoding="utf-8-sig", errors="ignore").strip() or None
    return None


for packed in Path("/kaggle/input").rglob("*.zip"):
    if packed.name.endswith("_outputs.zip"):
        continue  # an earlier training output, not the dataset
    target = Path(SCRATCH, "unzipped", packed.stem)
    if not target.exists():
        try:
            with zipfile.ZipFile(packed) as bundle:
                bundle.extractall(target)
        except zipfile.BadZipFile:
            print("skipped (not a zip):", packed)
roots = [Path("/kaggle/input"), Path(SCRATCH, "unzipped")]
folders = [folder for root in roots if root.exists() for folder in [root, *root.rglob("*")]
           if folder.is_dir() and not OWN_OUTPUT & set(folder.parts)]
found = max(folders, key=lambda folder: sum(caption_for(image) is not None
                                             for image in images_in(folder)), default=None)
images = images_in(found) if found is not None else []
captions = {image: caption_for(image) for image in images}
firsts = Counter(text.split(",")[0].strip() for text in captions.values() if text)
if len(images) < MIN_IMAGES or not firsts:
    fail(f"no dataset with at least {MIN_IMAGES} captioned images: attach the private dataset "
         "made by `ca lora prepare` (Add Input)")
TRIGGER = firsts.most_common(1)[0][0]
DATASET = Path(SCRATCH, "dataset")
shutil.rmtree(DATASET, ignore_errors=True)
DATASET.mkdir(parents=True)
fixed = []
for number, image in enumerate(images, start=1):
    text = captions[image] or f"{TRIGGER}, a man"
    if not text.lower().startswith(TRIGGER.lower()):
        text = f"{TRIGGER}, {text}"
    if text != captions[image]:
        fixed.append(image.name)
    stem = f"{number:02d}"
    shutil.copy(image, DATASET / f"{stem}{image.suffix.lower()}")
    Path(DATASET, f"{stem}.txt").write_text(text + "\n", encoding="utf-8")
JOB = f"{TRIGGER}_z_image_turbo"
print(f"Dataset: {found} -> {len(images)} images, trigger {TRIGGER!r}, job {JOB}")
if fixed:
    print("captions added or fixed for:", ", ".join(fixed))
save_status(dataset=str(found), images=len(images), trigger=TRIGGER, job=JOB,
            captions_fixed=fixed)

In [ ]:
# AI Toolkit at the pinned commit, then its requirements in ONE pip run with every version
# pinned. Kaggle's torch is held by a constraints file, so pip fails instead of replacing it.
import re
from importlib.metadata import PackageNotFoundError, version

TOOLKIT = Path(SCRATCH, "ai-toolkit")
if not (TOOLKIT / "run.py").exists():
    TOOLKIT.mkdir(parents=True, exist_ok=True)
    sh(f"git -C {TOOLKIT} init -q && git -C {TOOLKIT} remote add origin {TOOLKIT_REPO} && "
       f"git -C {TOOLKIT} fetch -q --depth 1 origin {TOOLKIT_COMMIT} && "
       f"git -C {TOOLKIT} checkout -q FETCH_HEAD")
commit = subprocess.run(["git", "-C", str(TOOLKIT), "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
if commit != TOOLKIT_COMMIT:
    fail(f"AI Toolkit is at {commit or 'nothing'}, not the pinned {TOOLKIT_COMMIT}")


def installed(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def requirement_lines(path: Path) -> list[str]:
    lines: list[str] = []
    for raw in path.read_text().splitlines():
        line = raw.split(" #", 1)[0].strip()
        if not line or line.startswith("#"):
            continue
        include = re.match(r"^(?:-r|--requirement)\s*=?\s*(\S+)", line)
        if include:
            lines += requirement_lines(path.parent / include.group(1))
        elif not line.startswith("-"):
            lines.append(line)
    return lines


specs: list[str] = []
unpinned: list[str] = []
for line in requirement_lines(TOOLKIT / "requirements.txt"):
    if line.startswith("git+"):
        specs.append(line)  # a source pinned to a commit by AI Toolkit itself
        continue
    plain = re.match(r"^([A-Za-z0-9][A-Za-z0-9._-]*)", line)
    name = plain.group(1).lower().replace("_", "-") if plain else line
    if name in SKIP_PACKAGES:
        continue
    if name not in PINNED:
        unpinned.append(line)
    else:
        specs.append(PINNED[name])
if unpinned:
    fail(f"requirements without a pin (add them to PINNED): {unpinned}")
specs += EXTRA_PINS
Path(SCRATCH, "pinned.txt").write_text("\n".join(specs) + "\n")
held = [f"{name}=={installed(name)}" for name in ("torch", "torchvision", "torchaudio")
        if installed(name)]
Path(SCRATCH, "constraints.txt").write_text("\n".join(held) + "\n")
print("holding:", held)
if installed("bitsandbytes"):
    sh(f"{sys.executable} -m pip uninstall -y -q bitsandbytes")
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{SCRATCH}/pinned.txt",
     "-c", f"{SCRATCH}/constraints.txt"], capture_output=True, text=True)
if install.returncode != 0:
    fail(f"pip could not install the pinned set: {(install.stdout + install.stderr)[-800:]}")
if installed("bitsandbytes"):
    fail("bitsandbytes came back as a dependency; it must not be installed")
VERSIONS = {name: installed(name) or "-" for name in (
    "torch", "torchao", "optimum-quanto", "diffusers", "transformers", "accelerate",
    "safetensors", "peft", "numpy", "scipy", "opencv-python")}
VERSIONS["python"] = sys.version.split()[0]
print("versions:", VERSIONS)
save_status(toolkit_commit=commit, versions=VERSIONS)

In [ ]:
# Z-Image as this AI Toolkit commit registers it, and the pinned Turbo training adapter.
from huggingface_hub import list_repo_files

if 'arch = "zimage"' not in (TOOLKIT / "extensions_built_in/diffusion_models/z_image/"
                             "z_image.py").read_text(errors="ignore"):
    fail("this AI Toolkit commit does not register the zimage arch")
ARCH = "zimage"
if ADAPTER_FILE not in list_repo_files(ADAPTER_REPO):
    fail(f"{ADAPTER_FILE} is not in {ADAPTER_REPO}")
ADAPTER = f"{ADAPTER_REPO}/{ADAPTER_FILE}"
print("Arch:", ARCH, "| adapter:", ADAPTER)
save_status(arch=ARCH, adapter=ADAPTER)

In [ ]:
# The LoRA to test: LORA_FILE if set, else the final file, else the highest step; from an
# attached training output (its folder or *_outputs.zip) or a dataset with the file. It must
# be finite and trained; its rank and step come from the file.
from safetensors import safe_open
from safetensors.torch import load_file

STEP_PATTERN = re.compile(r"_(\d+)\.safetensors$")
for packed in Path("/kaggle/input").rglob("*_outputs.zip"):
    target = Path(SCRATCH, "loras", packed.stem)
    if not target.exists():
        with zipfile.ZipFile(packed) as bundle:
            for member in bundle.namelist():
                if member.endswith(".safetensors"):
                    bundle.extract(member, target)
roots = [Path("/kaggle/input"), Path(SCRATCH, "loras")]
candidates = [path for root in roots if root.exists() for path in root.rglob("*.safetensors")
              if "smoke" not in path.name and "_set_aside" not in str(path)
              and "smoke_test" not in path.parts]
if LORA_FILE:
    chosen = [path for path in candidates if path.name == LORA_FILE]
else:
    finals = [path for path in candidates if not STEP_PATTERN.search(path.name)]
    chosen = finals or sorted(candidates, key=lambda path: int(
        STEP_PATTERN.search(path.name).group(1)))[-1:]  # type: ignore[union-attr]
if not chosen:
    fail(f"no LoRA found ({LORA_FILE or 'the final file'}): attach the training output "
         f"(found: {sorted({path.name for path in candidates}) or 'none'})")
LORA = chosen[0]
state = load_file(str(LORA))
if not all(bool(torch.isfinite(tensor.float()).all()) for tensor in state.values()):
    fail(f"{LORA.name} has NaN or infinite weights")
ups = [tensor for key, tensor in state.items() if "lora_B" in key or "lora_up" in key]
downs = [tensor for key, tensor in state.items() if "lora_A" in key or "lora_down" in key]
if not downs or (ups and not any(bool(tensor.abs().max() > 0) for tensor in ups)):
    fail(f"{LORA.name} is not a trained LoRA (no LoRA weights, or all up weights are zero)")
RANK = int(downs[0].shape[0])
with safe_open(str(LORA), framework="pt") as handle:
    meta = handle.metadata() or {}
try:
    LORA_STEP = int(json.loads(meta.get("training_info", "{}")).get("step", 0))
except (ValueError, TypeError):
    LORA_STEP = 0
del state
print(f"LoRA: {LORA} | rank {RANK} | step {LORA_STEP} | trigger {TRIGGER}")
save_status(lora=str(LORA), rank=RANK, lora_step=LORA_STEP)

In [ ]:
# The AI Toolkit config (YAML; JSON strings and numbers are valid YAML scalars). Every key is
# read by AI Toolkit's config classes at TOOLKIT_COMMIT. model.dtype and save.dtype default to
# float16 there, so both are set: the model, text encoder and VAE load in bf16.


def q(value: object) -> str:
    return json.dumps(value)


def write_config(path: Path, name: str, folder: str, *, steps: int, resolutions: list[int],
                 sample_size: int, prompts: list[str], every: int) -> None:
    listed = "\n".join(f"          - {q(prompt)}" for prompt in prompts)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(f"""job: extension
config:
  name: {q(name)}
  process:
    - type: "sd_trainer"
      training_folder: {q(folder)}
      device: "cuda:0"
      trigger_word: {q(TRIGGER)}
      network:
        type: "lora"
        linear: {RANK}
        linear_alpha: {RANK}
      save:
        dtype: {q(DTYPE)}
        save_every: {every}
        max_step_saves_to_keep: {KEEP_CHECKPOINTS}
      datasets:
        - folder_path: {q(str(DATASET))}
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          cache_latents_to_disk: true
          resolution: {q(resolutions)}
      train:
        batch_size: 1
        cache_text_embeddings: true
        skip_first_sample: true
        steps: {steps}
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: {q(OPTIMIZER)}
        lr: {LEARNING_RATE}
        dtype: {q(DTYPE)}
      model:
        name_or_path: {q(BASE_MODEL)}
        arch: {q(ARCH)}
        dtype: {q(DTYPE)}
        assistant_lora_path: {q(ADAPTER)}
        quantize: true
        qtype: "qfloat8"
        quantize_te: true
        qtype_te: "qfloat8"
        low_vram: true
      sample:
        sampler: "flowmatch"
        sample_every: {every}
        width: {sample_size}
        height: {sample_size}
        prompts:
{listed}
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 1
        sample_steps: 8
meta:
  name: "[name]"
  version: "1.0"
""")

In [ ]:
# Shared by the smoke test and the full run: progress parsing, memory, stopping.
import math
import signal

PROGRESS = re.compile(r"(\d+)/(\d+) \[[^\]]*?loss:\s*([-+0-9.eE]+|nan|inf)", re.IGNORECASE)


def losses(text: str) -> dict[int, str]:
    return {int(step): value.lower() for step, _, value in PROGRESS.findall(text)}


def bad(value: str) -> bool:
    return value in ("nan", "inf", "-inf") or not math.isfinite(float(value)) or (
        float(value) == 0.0)


def memory() -> str:
    info = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
    free = int(info["MemAvailable"].split()[0]) / 2**20
    total = int(info["MemTotal"].split()[0]) / 2**20
    gpu = subprocess.run(["nvidia-smi", "--id=0", "--query-gpu=memory.used,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    return f"RAM free {free:.1f}/{total:.1f} GB | GPU 0 {gpu or 'n/a'}"


def stop(process: subprocess.Popen[bytes], wait_s: int) -> None:
    os.killpg(process.pid, signal.SIGINT)  # AI Toolkit stops at Ctrl+C
    try:
        process.wait(timeout=wait_s)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)


def last_error(text: str) -> str:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    hits = [line for line in lines if re.search(r"error|exception|fatal|segmentation|killed",
                                                line, re.IGNORECASE)]
    return (hits or lines or ["no output"])[-1][-300:]

In [ ]:
# AI Toolkit's own sampler: the chosen LoRA is the newest checkpoint of a job whose steps equal
# its step, so no training step runs (and the learning rate is 0); the final sample pass makes
# one image per prompt, with "--m <strength> --seed <seed>" read by AI Toolkit per prompt.
from PIL import Image, ImageChops, ImageStat

TEST_NAME = f"{JOB}_test"
COUNT_PATTERN = re.compile(r"_(\d+)\.(?:jpg|jpeg|png|webp)$", re.IGNORECASE)


def run_sampler(folder: Path, prompts: list[str], size: int, minutes: int) -> list[Path]:
    """The images in prompt order; stops the notebook with the reason on any failure."""
    deadline = time.time() + minutes * 60
    shutil.rmtree(folder, ignore_errors=True)
    save_root = folder / TEST_NAME
    save_root.mkdir(parents=True)
    shutil.copy(LORA, save_root / f"{TEST_NAME}_{max(LORA_STEP, 1):09d}.safetensors")
    config = folder / f"{TEST_NAME}.yaml"
    write_config(config, TEST_NAME, str(folder), steps=max(LORA_STEP, 1),
                 resolutions=RESOLUTIONS, sample_size=size, prompts=prompts, every=10**6)
    log_path = folder / "log.txt"
    with log_path.open("w") as log:
        process = subprocess.Popen([sys.executable, "run.py", str(config)], cwd=TOOLKIT,
                                   stdout=log, stderr=subprocess.STDOUT, env=TRAIN_ENV,
                                   start_new_session=True)
        try:
            process.wait(timeout=max(60, deadline - time.time()))
        except subprocess.TimeoutExpired:
            stop(process, 60)
            fail(f"sampling {len(prompts)} image(s) took longer than {minutes} minutes")
    text = log_path.read_text(errors="ignore")
    if process.returncode != 0:
        fail(f"the sampler exited with code {process.returncode}: {last_error(text)}")
    images = sorted(
        (path for path in (save_root / "samples").glob("*") if COUNT_PATTERN.search(path.name)),
        key=lambda path: int(COUNT_PATTERN.search(path.name).group(1)),  # type: ignore[union-attr]
    )
    if len(images) != len(prompts):
        fail(f"expected {len(prompts)} image(s), the sampler made {len(images)} (see {log_path})")
    for path in images:
        with Image.open(path) as image:
            if max(ImageStat.Stat(image.convert("RGB")).stddev) < 5:
                fail(f"{path.name} is blank (NaN in sampling?)")
    return images


def flags(prompt: str, strength: float, seed: int) -> str:
    return f"{prompt} --m {strength} --seed {seed}"


first_prompt, first_seed = next(iter(PROMPTS.values()))
started = time.time()
run_sampler(Path(SCRATCH, "smoke"), [flags(first_prompt, 1.0, first_seed)], SMOKE_SIZE,
            SMOKE_MINUTES)
print(f"Smoke check passed in {(time.time() - started) / 60:.1f} min")
save_status(smoke_check="passed")

ORDER = [(name, strength) for strength in STRENGTHS for name in PROMPTS]
test_prompts = [flags(PROMPTS[name][0], strength, PROMPTS[name][1]) for name, strength in ORDER]
full = run_sampler(Path(SCRATCH, "full"), test_prompts, SIZE, FULL_MINUTES)
RESULTS: dict[tuple[str, float], Path] = {}
for (name, strength), path in zip(ORDER, full, strict=True):
    target = Path(OUTPUT_DIR, f"{name.replace(' ', '_')}_s{strength:.1f}{path.suffix}")
    shutil.copy(path, target)
    RESULTS[(name, strength)] = target
first_name = next(iter(PROMPTS))
with (
    Image.open(RESULTS[(first_name, STRENGTHS[0])]) as low,
    Image.open(RESULTS[(first_name, STRENGTHS[-1])]) as high,
):
    change = max(ImageStat.Stat(ImageChops.difference(low.convert("RGB"),
                                                      high.convert("RGB"))).mean)
if change < 1:
    fail(f"strength {STRENGTHS[0]} and {STRENGTHS[-1]} gave the same image: the LoRA "
         "strength was not applied")
print(f"Test set: {len(RESULTS)} images in {OUTPUT_DIR}")

In [ ]:
# One contact sheet (rows = prompts, columns = strengths) and one zip with everything.
from PIL import ImageDraw, ImageFont

CELL, LABEL, HEAD = 384, 170, 40
names = list(PROMPTS)
sheet = Image.new("RGB", (LABEL + CELL * len(STRENGTHS), HEAD + CELL * len(names)), "white")
draw = ImageDraw.Draw(sheet)
font = ImageFont.load_default(size=20)
for column, strength in enumerate(STRENGTHS):
    draw.text((LABEL + column * CELL + 10, 10), f"strength {strength}", fill="black", font=font)
for row, name in enumerate(names):
    draw.text((10, HEAD + row * CELL + CELL // 2), name, fill="black", font=font)
    for column, strength in enumerate(STRENGTHS):
        with Image.open(RESULTS[(name, strength)]) as image:
            sheet.paste(image.convert("RGB").resize((CELL, CELL)),
                        (LABEL + column * CELL, HEAD + row * CELL))
sheet.save(Path(OUTPUT_DIR, "contact_sheet.jpg"), quality=92)
settings = {"lora": LORA.name, "lora_step": LORA_STEP, "rank": RANK, "trigger": TRIGGER,
            "strengths": STRENGTHS, "size": SIZE, "sample_steps": 8, "guidance_scale": 1,
            "dtype": DTYPE, "toolkit_commit": TOOLKIT_COMMIT, "versions": VERSIONS,
            "prompts": {name: {"prompt": prompt.replace("[trigger]", TRIGGER), "seed": seed}
                        for name, (prompt, seed) in PROMPTS.items()}}
Path(OUTPUT_DIR, "settings.json").write_text(json.dumps(settings, indent=2))
save_status(status="done")
with zipfile.ZipFile(Path(OUTPUT_DIR, "lora_test.zip"), "w") as bundle:
    for path in sorted(Path(OUTPUT_DIR).iterdir()):
        if path.suffix != ".zip":
            bundle.write(path, path.name)
print("Contact sheet:", Path(OUTPUT_DIR, "contact_sheet.jpg"))
print("If he looks right: uv run ca lora set zimage", LORA.name)